In [1]:
import os
from collections import defaultdict
import pandas as pd
import numpy as np
from Bio import SeqIO

In [2]:
## define path
basedir = "/Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx"

# output directory
path_out = f"{basedir}/output/250909_4474_samples/final_test/"

# input files
# blastx output directory
path_in = f"{basedir}/data/250819_4474samples/blastout_all_protein/"
# fasta file
path_in_fasta = f"{basedir}/data/250819_4474samples/contig_viral_hit/"
# sample info table
sradata_path = f"{basedir}/data/Infection_Prediction_Stacking_0_sample_list_TN1000.txt"
# blacklist virus
blacklist_path = f"{basedir}/data/black_list_TN_ISG_low_new_format.txt"
# blastn output
path_in_blastn = f"{basedir}/data/250909_blastn/virus_hit_all_contigs_megablast.txt"

if not os.path.exists(path_out):
    os.mkdir(path_out)
print("saving files:", path_out)

saving files: /Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/250909_4474_samples/final_test/


In [3]:
def get_all_taxids(root_dir, takonkit_path):
    """
    extracts taxids from all samples
    """
    # initialize set
    taxids = set()

    # read blastx result file
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # to pass ".DS_store"
                continue

            # full path to each blast output file
            full_path = os.path.join(dirpath, filename)

            # read file
            col_name = [
                "qseqid", "sseqid", "pident", "length", "mismatch",
                "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
                "bitscore", "qframe", "staxids", "stitle", "qlen", 
                "slen", "qcovhsp", "scovhsp"
            ]

            if os.path.getsize(full_path) == 0:
                print("file size 0 ")
                continue

            one_sample = pd.read_table(full_path, header=None, dtype={13:str})  # staxids as string
            one_sample.columns = col_name

            # check if bitsocre exist for all queries
            if one_sample['bitscore'].isna().sum() > 0:
                print("NaN", filename)
            if (one_sample['bitscore'].dtype != np.float64) and (one_sample['bitscore'].dtype != int):
                print("dtype", filename, one_sample['bitscore'].dtype)
                print(one_sample['bitscore'])

            # sort by bitscore
            one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

            # delete duplicates
            one_sample_uniq = one_sample_sored.groupby("qseqid").first()

            # get taxid
            for line_taxid in one_sample_uniq["staxids"].to_list():
                if (line_taxid is None) or (line_taxid!=line_taxid):
                    continue
                raw_taxids = line_taxid.split(";")
                for i in range(len(raw_taxids)):
                    taxids.add(raw_taxids[i])

    # write taxid per line
    with open(takonkit_path, "w") as f:
        for taxid in sorted(list(taxids)):
            f.write(taxid + "\n")
    print(f"{len(taxids)} taxids in total")

    return taxids


def get_viral_hits(input_file, out_dir, lineage_info, blacklst):
    """
    saves only virus contigs based on taxid in the input table.
    """
    # num taxids with multiple groups
    num_ambiguous_hit = 0

    # confusion matrix group
    cm_group = input_file.split("/")[-2]

    # output file path
    output_folder = out_dir + "/taxonomy_output/"
    if not(os.path.exists(output_folder)):
        os.mkdir(output_folder)

    output_all_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.taxonomy.txt"
    output_viral_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.txt"
    output_viral_black_path = output_folder + "/" + cm_group + "." + input_file.split("/")[-1].split(".")[0] + ".blastx.virus.black.txt"

    # read file
    col_name = [
        "qseqid", "sseqid", "pident", "length", "mismatch",
        "gapopen", "qstart", "qend", "sstart", "send", "evalue", 
        "bitscore", "qframe", "staxids", "stitle", "qlen", "slen",
        "qcovhsp", "scovhsp"
    ]

    one_sample = pd.read_table(input_file, header=None, dtype={13:str})
    one_sample.columns = col_name

    # sort by bitscore
    one_sample_sored = one_sample.sort_values("bitscore", ascending=False)

    # delete duplicates
    one_sample_uniq = one_sample_sored.groupby("qseqid").first()

    # number of all contigs
    num_all_hits = len(set(one_sample_uniq.index.to_list()))

    # add taxonomy info
    one_sample_tax = one_sample_uniq.copy()
    one_sample_tax["staxids_lst"] = one_sample_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
    one_sample_tax["name_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
    one_sample_tax["group_lst"] = one_sample_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

    # add family and genus for the first taxid
    for idx in one_sample_tax.index:
        groups = one_sample_tax.loc[idx, "group_lst"]
        names = one_sample_tax.loc[idx, "name_lst"]
        if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
            num_ambiguous_hit += 1

    one_sample_tax["taxid_blastx"] = one_sample_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  # added on 2025/09/01
    one_sample_tax["family"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "family"] if((x is not None) and (x==x)) else x)
    one_sample_tax["genus"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "genus"] if((x is not None) and (x==x)) else x)
    one_sample_tax["species"] = one_sample_tax["staxids_lst"].apply(lambda x: lineage_info.loc[x[0], "species"] if((x is not None) and (x==x)) else x)

    # add tag if list has virus - virus = yes when all classification are virus
    one_sample_tax["is_virus"] = one_sample_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

    # extract only viral hit
    one_sample_virus = one_sample_tax.copy()
    one_sample_virus = one_sample_virus[one_sample_virus["is_virus"] == 1]

    # number of viral contigs
    num_viral_hits = len(set(one_sample_virus.index.to_list()))

    # read black list
    blackvirus = []
    with open(blacklst, "r") as f_in:
        for line in f_in:
            blackvirus.append(line.strip())

    # mask virus in black list
    one_sample_black_removed = one_sample_virus.copy()
    one_sample_black_removed = one_sample_black_removed[~one_sample_black_removed["family"].isin(blackvirus)]

    # number of viral contigs after filtering out black list virus
    num_viral_hits_after_filter = len(set(one_sample_black_removed.index.to_list()))

    # save
    one_sample_tax.to_csv(output_all_path, sep="\t")
    one_sample_virus.to_csv(output_viral_path, sep="\t")
    one_sample_black_removed.to_csv(output_viral_black_path, sep="\t")

    return one_sample_black_removed, num_all_hits, num_viral_hits, num_viral_hits_after_filter, num_ambiguous_hit


def count_viral_contigs(path_in: str, path_out, lineage_info, blacklist_path, infection_th=1):
    """
    counts virus contigs by CM group.
    """
    # initalize dictionary
    cm_hits = defaultdict(list)
    cm_props = defaultdict(list)
    cm_class = defaultdict(list)
    sample_class = defaultdict(lambda: defaultdict(str))

    # initialize flag variable
    num_amb_hit = 0
    is_fist_file = 1

    # read all files
    for dirpath, dirnames, filenames in os.walk(path_in):
        for filename in filenames:
            if not (filename.endswith(".txt")):  # to get rid of ".DS_store"
                continue

            # full path to each blast output file
            full_path = os.path.join(dirpath, filename)

            if os.path.getsize(full_path) == 0:
                print("filesize 0")
                continue

            # get and save viral contigs
            df_one_sample, num_all_contigs, num_viral_contigs, num_viral_black, num_amb_hit_each = get_viral_hits(full_path, path_out, lineage_info, blacklist_path)
            num_amb_hit += num_amb_hit_each

            # store number of viral contigs per cm group
            cm_group = full_path.split('/')[-2]
            cm_hits[cm_group].append(num_viral_black)
            cm_props[cm_group].append(num_viral_black / num_all_contigs)
    
            # infection 0 or 1 (use threshold)
            if num_viral_black >= infection_th:
                cm_class[cm_group].append("1")
            else:
                cm_class[cm_group].append("0")

            # concatnate results
            if is_fist_file == 1:
                df_all = df_one_sample.copy()
                is_fist_file = 0
            else:
                df_all = pd.concat([df_all, df_one_sample])

            # make info table per sample
            sample = full_path.split('/')[-1].split(".")[0].split("_")[0]
            if num_viral_black >= infection_th:
                sample_class[sample]["viral_contig"] = "1"
            else:
                sample_class[sample]["viral_contig"] = "0"
            sample_class[sample]["cm"] = cm_group

    print("Virus ambiguous hit:", num_amb_hit)
    print("=======")

    for k, v in cm_hits.items():
        print(k, np.mean(v))
    for k, v in cm_props.items():
        print(k, f"{np.mean(v):.2}")

    print("=======")
    return cm_hits, cm_props, cm_class, sample_class, df_all


def collect_from_tsv(tsv_file, output_file):
    """
    Read fasta path + contig IDs from a TSV and write selected sequences
    into a single multi-FASTA file.

    Parameters:
        tsv_file (str): Path to TSV file with columns [path, qseqid]
        output_file (str): Path to output fasta file
    """
    df = pd.read_table(tsv_file, sep="\t")
    with open(output_file, "w") as out_f:
        for fasta_file in sorted(list(set(df["path"].to_list()))):
            record_dict = SeqIO.index(fasta_file, "fasta")
            contigs = df.loc[df["path"] == fasta_file, "qseqid"].to_list()
            for contig in contigs:
                if contig in record_dict:
                    SeqIO.write(record_dict[contig], out_f, "fasta")
                else:
                    print(f"[Warning] {contig} not found in {fasta_file}")


def get_seq_from_fasta(fasta_file, contig_name):
    """
    Return the sequence string for a given contig name from a fasta file.

    Parameters:
        fasta_file (str): Path to fasta file
        contig_name (str): Contig ID (matches the part after '>' up to first space)

    Returns:
        str: Sequence as a string, or None if not found
    """
    record_dict = SeqIO.index(fasta_file, "fasta")
    if contig_name in record_dict:
        return record_dict[contig_name].seq
    else:
        return None

# Taxonkit

In [4]:
# get all taxids
taxids = get_all_taxids(path_in, path_out + "taxids.txt")

17083 taxids in total


Run taxonkit (`Fig5_table_taxonkit.sh`) in `path_out` directory.

# Phage

In [5]:
# read taxonkit output
taxinfo = pd.read_table(path_out + "taxids_long.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# add taxonomy - {d};{K};{p};{c};{o};{f};{g};{s}
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# count NaN
print("#NaN:", taxinfo.isna().sum().sum())
# fill NaN
taxinfo = taxinfo.fillna("-")

# size
print("taxonkit output original size:", taxinfo.shape[0])

# find Caudoviricetes, phage, 
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    lineage = taxinfo.loc[idx, "lineage"]
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. phage sp
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# phage group order
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# size
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# save table
taxinfo.to_csv(path_out + "taxids_phage.tsv", sep="\t")

# save phage list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# show
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 18
taxonkit output original size: 17083
phage order: ['Crassvirales', 'unclassified Caudoviricetes order', 'unclassified Viruses order']
Caudoviricetes: 676
phage: 306
 * not Caudoviricetes: 8
Inoviridae: 1
Lavidaviridae: 0
Microviridae: 1


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae
5937,1868660,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured Mediterranean phage,1,0,1,0,0,0
9572,2740180,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Streptomyces phage Dagobah,1,0,1,0,0,0
9813,278008,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured phage,1,0,1,0,0,0
11061,2961709,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Salmonella phage vB_Sal_PHB48,1,0,1,0,0,0
11477,3034680,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Staphylococcus phage Southeast,1,0,1,0,0,0
11847,3093841,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Bacillus phage CM1,1,0,1,0,0,0
12724,38018,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,Bacteriophage sp.,1,0,1,0,0,0
15498,707152,unclassified Viruses domain;unclassified Virus...,unclassified Viruses domain,unclassified Viruses kingdom,unclassified Viruses phylum,unclassified Viruses class,unclassified Viruses order,unclassified Viruses family,unclassified Viruses genus,uncultured marine phage,1,0,1,0,0,0


# Add taxonomy

In [6]:
# read taxonkit output
taxid_in = path_out + "taxids_lineage.txt"
lineage_info = pd.read_table(taxid_in, header=None)
lineage_info.columns = ["taxid", "domain", "fgs"]

# check NaN
print("#NaN beginning:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# add family, genus, speceis column
lineage_info["taxid"] = lineage_info["taxid"].astype(str)
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)  # if nan
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)  # if nan
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)  # if nan

# replace domain with phage for phage group
phage_ids = []
with open(path_out + "phage_list.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# check NaN
print("#NaN end:", lineage_info.isna().sum().sum())
nan_rows =lineage_info[lineage_info.isna().any(axis=1)]
print(nan_rows)
print()

# fill NaN
lineage_info = lineage_info.fillna("-")

print(f"{lineage_info.shape[0]} taxids in total\n")

# show example hit for each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# show
lineage_info

#NaN beginning: 4
        taxid domain  fgs
1426  1230063    NaN  NaN
1429  1230068    NaN  NaN

#NaN end: 10
        domain  fgs family genus species
taxid                                   
1230063    NaN  NaN    NaN   NaN     NaN
1230068    NaN  NaN    NaN   NaN     NaN

17083 taxids in total

All domain: {'phage_group', '-', 'unclassified other entries domain', 'unclassified Viruses domain', 'Eukaryota', 'unclassified unclassified entries domain', 'Bacteria', 'Archaea'} 

##### phage_group ####
size: 686
              domain             species
taxid                                   
1029988  phage_group   Erskinevirus EaH2
103216   phage_group     Peduovirus Wphi
1051675  phage_group  Waedenswilvirus S6

##### - ####
size: 2
        domain species
taxid                 
1230063      -       -
1230068      -       -

##### unclassified other entries domain ####
size: 550
                                    domain  \
taxid                                        
1042257  unclassifi

,domain,fgs,family,genus,species
taxid,,,,,
100,Bacteria,Xanthobacteraceae;Ancylobacter;Ancylobacter aq...,Xanthobacteraceae,Ancylobacter,Ancylobacter aquaticus
100035,Eukaryota,unclassified Pleosporales family;Massariosphae...,unclassified Pleosporales family,Massariosphaeria,Massariosphaeria phaeospora
1000413,Eukaryota,Sapindaceae;Acer;Acer yangbiense,Sapindaceae,Acer,Acer yangbiense
1000504,unclassified Viruses domain,Orthomyxoviridae;Alphainfluenzavirus;Alphainfl...,Orthomyxoviridae,Alphainfluenzavirus,Alphainfluenzavirus influenzae
1000590,Bacteria,Staphylococcaceae;Staphylococcus;Staphylococcu...,Staphylococcaceae,Staphylococcus,Staphylococcus epidermidis
...,...,...,...,...,...
999809,Eukaryota,Ustilaginaceae;Sporisorium;Sporisorium reilianum,Ustilaginaceae,Sporisorium,Sporisorium reilianum
999810,Eukaryota,Sclerotiniaceae;Botrytis;Botrytis cinerea,Sclerotiniaceae,Botrytis,Botrytis cinerea
999883,unclassified Viruses domain,Marseilleviridae;unclassified Marseilleviridae...,Marseilleviridae,unclassified Marseilleviridae genus,Lausannevirus


# Extract viral tophit

In [7]:
# extract virus tophit
infection_threshod = 1
cm_hits, cm_props, cm_class, sample_class, df_all = count_viral_contigs(path_in, path_out, lineage_info, blacklist_path, infection_threshod)

# viral contig table
df = df_all.copy().reset_index()
df["sample"] = df["qseqid"].apply(lambda x: x.split("_")[0])
# cm group
sampledf = pd.DataFrame(sample_class).T
df = df.merge(sampledf, left_on = "sample", right_index=True, how="left")

# add count of viral contigs per sample
virus_hit_count = pd.DataFrame(df.value_counts("sample"))
df = df.merge(virus_hit_count, on="sample", how="left")
df = df.sort_values("sample")
df = df.sort_values("count", ascending=False)

# count NAN
print("#NaN:", df.isna().sum().sum())

# save
df.to_csv(path_out + "virus_all.txt", sep="\t")

Virus ambiguous hit: 2530
FP 0.8367911479944675
TP 10.783582089552239
FN 8.288686605981795
TN 0.384
FP 0.0014
TP 0.016
FN 0.013
TN 0.00078
#NaN: 0


# BLASTn

In [8]:
## blastn
# create viral contig + path table
df = pd.read_table(path_out + "virus_all.txt", index_col=0)
df = df.drop("viral_contig", axis=1)
df_fasta = df[["qseqid", "cm", "sample"]].copy()
for idx in df_fasta.index:
    df_fasta.loc[idx, "path"] = f"{path_in_fasta}{df_fasta.loc[idx, "cm"]}/{df_fasta.loc[idx, "sample"]}.viral_hit_contigs.fa"
df_fasta = df_fasta[["path", "qseqid"]]
# save table
df_fasta.to_csv(path_out + "virus_contig_path.txt", sep="\t", index=False)
# create fasta file of viral contigs
collect_from_tsv(path_out + "virus_contig_path.txt", path_out + "virus_hit_all_contigs.fasta")

Run BLASTn (script: `Fig5_table_blastn.sh`, output file name: `path_in_blastn`)

# Taxonkit for blastn

In [9]:
## taxonkit
# path for taxonkit result
takonkit_path_2 = path_out + "taxids_blastn.txt"

# initialize set
taxids = set()

# read blastn result
col_name= [
    "qseqid", "sseqid", "pident", "length", 
    "mismatch", "gapopen", "qstart", "qend", 
    "sstart", "send", "evalue", "bitscore", 
    "qframe", "staxids", "stitle", "qlen", 
    "slen", "qcovs", "qcovhsp"
]

blastn = pd.read_table(path_in_blastn, header=None, dtype={13:str})  # staxids as string
blastn.columns = col_name
print("blastn output original size:", blastn.shape)
print("#NaN:", blastn.isna().sum().sum())

# sort by bitscore
blastn_sorted = blastn.sort_values("bitscore", ascending=False)
# delete duplicates
blastn_uniq = blastn_sorted.groupby("qseqid").first()
print("blastn output size after choosing tophits:", blastn_uniq.shape)

# get taxid
for line_taxid in blastn_uniq["staxids"].to_list():
    if (line_taxid is None) or (line_taxid!=line_taxid):
        continue
    raw_taxids = line_taxid.split(";")
    for i in range(len(raw_taxids)):
        taxids.add(raw_taxids[i])

# write taxid per line
with open(takonkit_path_2, "w") as f:
    for taxid in taxids:
        f.write(taxid + "\n")

print(f"{len(taxids)} taxids in total")

# show
blastn_uniq.head(10)

blastn output original size: (2914726, 19)
#NaN: 0
blastn output size after choosing tophits: (11981, 18)
1035 taxids in total


,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qframe,staxids,stitle,qlen,slen,qcovs,qcovhsp
qseqid,,,,,,,,,,,,,,,,,,
ERR10662039_124963,gi|2118777509|gb|MZ367372.1|,95.794,214,9,0,1,214,5756,5543,8.140000e-91,346.0,1,336959,Chicken astrovirus isolate CAV/Belgium/4134_00...,215,7480,99,99
ERR10662039_29322,gi|389618860|gb|JN582313.1|,94.014,852,51,0,1,852,1349,2200,0.000000e+00,1291.0,1,336959,Chicken astrovirus isolate VF06-7/3 capsid pro...,852,2229,100,100
ERR10662039_58854,gi|2118777509|gb|MZ367372.1|,95.690,464,20,0,1,464,5793,6256,0.000000e+00,747.0,1,336959,Chicken astrovirus isolate CAV/Belgium/4134_00...,464,7480,100,100
ERR10662039_61653,gi|2118777509|gb|MZ367372.1|,98.870,354,4,0,1,354,3854,3501,9.850000e-177,632.0,1,336959,Chicken astrovirus isolate CAV/Belgium/4134_00...,354,7480,100,100
ERR10662039_85530,gi|389618860|gb|JN582313.1|,96.438,365,13,0,1,365,178,542,8.040000e-168,603.0,1,336959,Chicken astrovirus isolate VF06-7/3 capsid pro...,367,2229,99,99
ERR11505140_111036,gi|2414813051|ref|XR_008221636.1|,98.565,209,3,0,1,209,4996,5204,4.670000e-98,370.0,1,564181,PREDICTED: Peromyscus californicus insignis un...,209,6015,100,100
ERR11505145_2536,gi|1908055260|ref|XR_004943307.1|,99.842,2528,4,0,1,2528,3396,5923,0.000000e+00,4647.0,1,38674,PREDICTED: Onychomys torridus uncharacterized ...,2613,5923,97,97
ERR11505147_9442,gi|1908055261|ref|XR_004943308.1|,98.251,1887,24,3,1,1880,5459,3575,0.000000e+00,3293.0,1,38674,PREDICTED: Onychomys torridus uncharacterized ...,1880,6013,100,100
ERR11505154_3122,gi|1908055259|ref|XR_004943306.1|,99.820,3883,7,0,1,3883,2065,5947,0.000000e+00,7132.0,1,38674,PREDICTED: Onychomys torridus uncharacterized ...,3883,5955,100,100


Run `Fig5_table_taxonkit_blastn.sh` in `path_out` directory.

# Phage for blastn

In [10]:
## phage list
# read taxonkit output
taxinfo = pd.read_table(path_out + "taxids_long_blastn.txt", header=None, dtype={0:str})
taxinfo.columns = ["taxid", "lineage"]

# add taxonomy - {d};{K};{p};{c};{o};{f};{g};{s}
taxinfo["domain"] = taxinfo["lineage"].apply(lambda x: x.split(";")[0] if x == x else x)
taxinfo["kingdom"] = taxinfo["lineage"].apply(lambda x: x.split(";")[1] if x == x else x)
taxinfo["phylum"] = taxinfo["lineage"].apply(lambda x: x.split(";")[2] if x == x else x)
taxinfo["class"] = taxinfo["lineage"].apply(lambda x: x.split(";")[3] if x == x else x)
taxinfo["order"] = taxinfo["lineage"].apply(lambda x: x.split(";")[4] if x == x else x)
taxinfo["family"] = taxinfo["lineage"].apply(lambda x: x.split(";")[5] if x == x else x)
taxinfo["genus"] = taxinfo["lineage"].apply(lambda x: x.split(";")[6] if x == x else x)
taxinfo["species"] = taxinfo["lineage"].apply(lambda x: x.split(";")[7] if x == x else x)

# count NaN
print("#NaN:", taxinfo.isna().sum().sum())
# fill NaN
taxinfo = taxinfo.fillna("-")
# size
print("taxonkit result size:", taxinfo.shape[0])

# find Caudoviricetes, phage, Inoviridae, Lavidaviridae, Microviridae
taxinfo["is_phage"] = 0
taxinfo["r1_Caudoviricetes"] = 0
taxinfo["r2_includes_phage"] = 0
taxinfo["r3_Inoviridae"] = 0
taxinfo["r4_Lavidaviridae"] = 0
taxinfo["r5_Microviridae"] = 0
phage_group_order = set()

for idx in taxinfo.index:
    lineage = taxinfo.loc[idx, "lineage"]
    # 1. Caudoviricetes
    if taxinfo.loc[idx, "class"] == "Caudoviricetes":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r1_Caudoviricetes"] = 1
    # 2. phage sp
    if ("phage" in taxinfo.loc[idx, "species"]) and (taxinfo.loc[idx, "domain"] == 'unclassified Viruses domain'):
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r2_includes_phage"] = 1
        phage_group_order.add(taxinfo.loc[idx, "order"])
    # 3. Inoviridae
    if taxinfo.loc[idx, "family"] == "Inoviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r3_Inoviridae"] = 1
    # 4. Lavidaviridae
    if taxinfo.loc[idx, "family"] == "Lavidaviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r4_Lavidaviridae"] = 1
    # 5. Microviridae
    if taxinfo.loc[idx, "family"] == "Microviridae":
        taxinfo.loc[idx, "is_phage"] = 1
        taxinfo.loc[idx, "r5_Microviridae"] = 1

# phage group order
phage_group_order = sorted(list(phage_group_order))
print("phage order:", phage_group_order)

# size
print("Caudoviricetes:", taxinfo.loc[taxinfo["r1_Caudoviricetes"] == 1, :].shape[0])
print("phage:", taxinfo.loc[taxinfo["r2_includes_phage"] == 1, :].shape[0])
print(" * not Caudoviricetes:", taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :].shape[0])
print("Inoviridae:", taxinfo.loc[taxinfo["r3_Inoviridae"] == 1, :].shape[0])
print("Lavidaviridae:", taxinfo.loc[taxinfo["r4_Lavidaviridae"] == 1, :].shape[0])
print("Microviridae:", taxinfo.loc[taxinfo["r5_Microviridae"] == 1, :].shape[0])

# save table
taxinfo.to_csv(path_out + "taxids_phage.tsv", sep="\t")

# save phage list
phages = taxinfo.loc[taxinfo["is_phage"] == 1, "taxid"].to_list()
with open(path_out + "phage_list_blastn.txt", "w") as f:
    for pid in phages:
        f.write(pid + "\n")

# show
taxinfo.loc[(taxinfo["r1_Caudoviricetes"] == 0) & (taxinfo["r2_includes_phage"] == 1), :]

#NaN: 9
taxonkit result size: 1035
phage order: []
Caudoviricetes: 0
phage: 0
 * not Caudoviricetes: 0
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 1


,taxid,lineage,domain,kingdom,phylum,class,order,family,genus,species,is_phage,r1_Caudoviricetes,r2_includes_phage,r3_Inoviridae,r4_Lavidaviridae,r5_Microviridae


In [11]:
## add lineage info to taxid list
# input path
taxid_in = path_out + "taxids_lineage_blastn.txt"

# read lineage table
lineage_info = pd.read_table(taxid_in, header=None, dtype={0:str})
lineage_info.columns = ["taxid", "domain", "fgs"]

# supplyment 3414891 (deleted)
lineage_info.loc[lineage_info["taxid"]=="3414891", "domain"] = "unclassified Viruses domain"
lineage_info.loc[lineage_info["taxid"]=="3414891", "fgs"] = "unclassified Bunyavirales;unclassified Bunyavirales;unclassified Bunyavirales"

# check NaN
print("#NaN:", lineage_info.isna().sum().sum())

# add family, genus, speceis columns
lineage_info = lineage_info.set_index("taxid")
lineage_info["family"] = lineage_info["fgs"].apply(lambda x: x.split(";")[0] if x==x else np.nan)
lineage_info["genus"] = lineage_info["fgs"].apply(lambda x: x.split(";")[1] if x==x else np.nan)
lineage_info["species"] = lineage_info["fgs"].apply(lambda x: x.split(";")[2] if x==x else np.nan)


# replace domain with phage for phage group
phage_ids = []
with open(path_out + "phage_list_blastn.txt", "r") as f:
    for line in f:
        line = line.strip()
        phage_ids.append(line)

for idx in lineage_info.index:
    if idx in phage_ids:
        lineage_info.loc[idx, "domain"] = "phage_group"

# fill NaN
lineage_info = lineage_info.fillna("-")

# size
print(f"{lineage_info.shape[0]} taxids in total\n")

# show example hit for each domain group
print("All domain:", set(lineage_info["domain"].to_list()), "\n")
for domain in set(lineage_info["domain"].to_list()):
    print(f"##### {domain} ####")
    print(f"size: {lineage_info[lineage_info["domain"] == domain].shape[0]}")
    print(lineage_info[["domain", "species"]][lineage_info["domain"] == domain].head(3))
    print()

# show
lineage_info

#NaN: 0
1035 taxids in total

All domain: {'phage_group', 'unclassified other entries domain', 'unclassified Viruses domain', 'Eukaryota', 'Bacteria'} 

##### phage_group ####
size: 1
              domain                   species
taxid                                         
2929290  phage_group  Dipodfec virus RodF1_123

##### unclassified other entries domain ####
size: 6
                                    domain                  species
taxid                                                              
32630    unclassified other entries domain      synthetic construct
3425667  unclassified other entries domain        Vector PEMC-tsSeV
2268895  unclassified other entries domain  Cloning vector pMDT7GX1

##### unclassified Viruses domain ####
size: 968
                              domain                           species
taxid                                                                 
1348384  unclassified Viruses domain  Rotavirus aspergastroenteritidis
12071    unclassif

,domain,fgs,family,genus,species
taxid,,,,,
1348384,unclassified Viruses domain,Sedoreoviridae;Rotavirus;Rotavirus aspergastro...,Sedoreoviridae,Rotavirus,Rotavirus aspergastroenteritidis
4690,Eukaryota,Liliaceae;Lilium;Lilium longiflorum,Liliaceae,Lilium,Lilium longiflorum
12071,unclassified Viruses domain,Picornaviridae;Enterovirus;Enterovirus B,Picornaviridae,Enterovirus,Enterovirus B
11049,unclassified Viruses domain,Arteriviridae;Betaarterivirus;Betaarterivirus ...,Arteriviridae,Betaarterivirus,Betaarterivirus suid 1
2809412,unclassified Viruses domain,Picornaviridae;Gallivirus;Gallivirus sp.,Picornaviridae,Gallivirus,Gallivirus sp.
...,...,...,...,...,...
2844800,unclassified Viruses domain,Anelloviridae;Gyrovirus;Gyrovirus galga1,Anelloviridae,Gyrovirus,Gyrovirus galga1
1813615,unclassified Viruses domain,unclassified Picornavirales family;Husavirus;H...,unclassified Picornavirales family,Husavirus,Husa-like virus KS-2016a
1245563,unclassified Viruses domain,Picornaviridae;Sapelovirus;WUHARV Sapelovirus 2,Picornaviridae,Sapelovirus,WUHARV Sapelovirus 2


In [12]:
## merge blastn result and taxomoy info
# read blastn result
blastn_tax = blastn_uniq.copy()
blastn_tax["staxids_lst"] = blastn_tax["staxids"].apply(lambda x: x.split(";") if((x is not None) and (x==x)) else x)
blastn_tax["name_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "fgs"] for taxid in x] if((x is not None) and (x==x)) else x)
blastn_tax["group_lst"] = blastn_tax["staxids_lst"].apply(lambda x: [lineage_info.loc[taxid, "domain"] for taxid in x] if((x is not None) and (x==x)) else x)

# add taxonomy lineage
num_ambiguous_hit = 0
for idx in blastn_uniq.index:
    groups = blastn_tax.loc[idx, "group_lst"]
    names = blastn_tax.loc[idx, "name_lst"]
    # count hits with taxids with multiple groups
    if ((groups is not None) and (groups == groups)) and any("Viruses" in item for item in groups) and (len(set(groups)) > 1):
        num_ambiguous_hit += 1
        print("#### Different root ####")
        print(groups)
        print(names)
        print()
print("Num ambiguou hits:", num_ambiguous_hit)

# add family, genus, species
blastn_tax["taxid"] = blastn_tax["staxids_lst"].apply(lambda x: x[0] if((x is not None) and (x==x)) else x)  # added on 2025/09/01
blastn_tax["family"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "family"] if((x is not None) and (x==x)) else x)
blastn_tax["genus"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "genus"] if((x is not None) and (x==x)) else x)
blastn_tax["species"] = blastn_tax["taxid"].apply(lambda x: lineage_info.loc[x, "species"] if((x is not None) and (x==x)) else x)

# check NaN
print("#NaN:", blastn_tax.isna().sum().sum())

# add tag if list has virus - virus = yes when all classification are virus
blastn_tax["is_virus"] = blastn_tax["group_lst"].apply(lambda x: 1 if((x is not None) and (x==x)) and all("Viruses" in item for item in x) else 0)

# edit columns name
blastn_tax.columns = blastn_tax.columns.map(lambda x: x + "_blastn")  # add "_blastn" to column names

# save
blastn_tax.to_csv(path_out + "blastn_result_top_hit.tsv", sep="\t")

# show
print("Size after adding taxonomy:", blastn_tax.shape)
blastn_tax.head(10)

Num ambiguou hits: 0
#NaN: 0
Size after adding taxonomy: (11981, 26)


,sseqid_blastn,pident_blastn,length_blastn,mismatch_blastn,gapopen_blastn,qstart_blastn,qend_blastn,sstart_blastn,send_blastn,evalue_blastn,...,qcovs_blastn,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn
qseqid,,,,,,,,,,,,,,,,,,,,,
ERR10662039_124963,gi|2118777509|gb|MZ367372.1|,95.794,214,9,0,1,214,5756,5543,8.140000e-91,...,99,99,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_29322,gi|389618860|gb|JN582313.1|,94.014,852,51,0,1,852,1349,2200,0.000000e+00,...,100,100,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_58854,gi|2118777509|gb|MZ367372.1|,95.690,464,20,0,1,464,5793,6256,0.000000e+00,...,100,100,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_61653,gi|2118777509|gb|MZ367372.1|,98.870,354,4,0,1,354,3854,3501,9.850000e-177,...,100,100,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR10662039_85530,gi|389618860|gb|JN582313.1|,96.438,365,13,0,1,365,178,542,8.040000e-168,...,99,99,[336959],[Astroviridae;Avastrovirus;Chicken astrovirus],[unclassified Viruses domain],336959,Astroviridae,Avastrovirus,Chicken astrovirus,1
ERR11505140_111036,gi|2414813051|ref|XR_008221636.1|,98.565,209,3,0,1,209,4996,5204,4.670000e-98,...,100,100,[564181],[Cricetidae;Peromyscus;Peromyscus californicus],[Eukaryota],564181,Cricetidae,Peromyscus,Peromyscus californicus,0
ERR11505145_2536,gi|1908055260|ref|XR_004943307.1|,99.842,2528,4,0,1,2528,3396,5923,0.000000e+00,...,97,97,[38674],[Cricetidae;Onychomys;Onychomys torridus],[Eukaryota],38674,Cricetidae,Onychomys,Onychomys torridus,0
ERR11505147_9442,gi|1908055261|ref|XR_004943308.1|,98.251,1887,24,3,1,1880,5459,3575,0.000000e+00,...,100,100,[38674],[Cricetidae;Onychomys;Onychomys torridus],[Eukaryota],38674,Cricetidae,Onychomys,Onychomys torridus,0
ERR11505154_3122,gi|1908055259|ref|XR_004943306.1|,99.820,3883,7,0,1,3883,2065,5947,0.000000e+00,...,100,100,[38674],[Cricetidae;Onychomys;Onychomys torridus],[Eukaryota],38674,Cricetidae,Onychomys,Onychomys torridus,0


# Save

In [13]:
## merge blastn result and blastx result
# read blastx result
df_all = pd.read_table(path_out + "virus_all.txt", index_col=0)

# read blastn result
df_blastn = pd.read_table(path_out + "blastn_result_top_hit.tsv")
print("df_blastn shape:", df_blastn.shape)

# merge blastn and blastx
df_all_blastn = df_all.merge(df_blastn, on="qseqid", how="left")
print("df_all_blastn shape:", df_all_blastn.shape)
print("#NaN", df_all_blastn.isna().sum().sum())

# add query seq
df_all_blastn["query_seq"] = df_all_blastn.apply(lambda row: get_seq_from_fasta(f"{path_in_fasta}{row["cm"]}/{row["sample"]}.viral_hit_contigs.fa", row["qseqid"]), axis=1)

# save
df_all_blastn.to_csv(path_out + "virus_all.blastn.tsv", sep="\t")
# show
df_all_blastn

df_blastn shape: (11981, 27)
df_all_blastn shape: (14353, 57)
#NaN 61672


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,qcovhsp_blastn,staxids_lst_blastn,name_lst_blastn,group_lst_blastn,taxid_blastn,family_blastn,genus_blastn,species_blastn,is_virus_blastn,query_seq
0,SRR8445393_281302,QMW68894.1,71.2,52,15,0,158,3,183,234,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(A, C, C, C, G, A, A, A, T, T, T, A, T, A, G, ..."
1,SRR8445393_357285,UNY42121.1,51.9,54,23,1,163,11,1483,1536,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(G, C, T, C, T, T, C, C, G, A, T, C, T, A, G, ..."
2,SRR8445393_136264,QZW33720.1,97.9,190,4,0,610,41,96,285,...,99.0,['2748378'],['unclassified Cressdnaviricota family;unclass...,['unclassified Viruses domain'],2748378.0,unclassified Cressdnaviricota family,unclassified Cressdnaviricota genus,Cressdnaviricota sp.,1.0,"(C, G, G, C, A, T, G, T, C, T, C, A, T, T, G, ..."
3,SRR8445393_141616,QTE03397.1,62.3,53,17,1,160,11,10,62,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(C, C, A, C, G, C, A, G, T, A, G, T, T, A, A, ..."
4,SRR8445393_160709,QTE03459.1,49.2,61,26,1,190,8,88,143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(C, T, C, T, T, C, C, G, A, T, C, T, T, G, G, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14348,SRR24422863_7911,YP_010087183.1,61.7,162,62,0,6,491,172,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(C, A, C, A, G, A, G, C, T, G, G, C, A, G, A, ..."
14349,SRR24422864_12018,YP_010087183.1,64.7,133,47,0,1,399,201,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(A, T, C, A, G, T, T, T, A, G, A, G, G, C, T, ..."
14350,SRR24422865_31476,YP_010087183.1,63.6,151,55,0,663,1115,183,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(G, A, C, A, T, T, G, A, C, T, T, T, G, G, C, ..."
14351,SRR24422866_55522,YP_010087183.1,61.8,165,63,0,485,979,169,333,...,100.0,['564181'],['Cricetidae;Peromyscus;Peromyscus californicus'],['Eukaryota'],564181.0,Cricetidae,Peromyscus,Peromyscus californicus,0.0,"(T, G, C, A, T, G, G, A, A, T, T, T, T, G, A, ..."
